# School Locations: Urban Informatics Final Project

Spring 2026

**Contributors:** Anwar Baroudi, Phoebe Chiu, Noelani Fixler, Michael Huang

### **1. Setting Up the Document**

In [1]:
# Import the libraries and modules we need
import pandas as pd
import numpy as np
import math
import os
import geopandas as gpd
from tabulate import tabulate
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **2. Organizing Equity Priority Communities Data**

In [4]:
# Get the Census tracts for all Equity Priority Communities
# Source: https://opendata.mtc.ca.gov/datasets/equity-priority-communities-plan-bay-area-2050/explore?location=37.727536%2C-122.235343%2C12

'''
Note: There are will be code strings for each person's unique filepaths. Hide other people's filepath code as a comment by using Ctrl + /.

'''

# Michael's filepath
# mtc_epcs = gpd.read_file("/content/drive/MyDrive/Urban Informatics Final Project/Data/Equity Priority Communities/Unprocessed EPC Data/communities_of_concern_2020_acs2018_8293892476282991523")

# Phoebe's filepath
phoebe_filepath_epcs = '/content/drive/MyDrive/Coursework/Spring 2026/CP 255/Urban Informatics Final Project/Data/Equity Priority Communities/Unprocessed EPC Data/communities_of_concern_2020_acs2018_8293892476282991523/equity_priority_communities_2020_acs2018.shp'
mtc_epcs = gpd.read_file(phoebe_filepath_epcs)


# Clean column names
mtc_epcs.columns = (
    mtc_epcs.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [5]:
# Call the first five rows of the Census tract data
mtc_epcs.head(2)

,geoid,state_fip,county_fip,tract,tot_pop,tot_pop_po,tot_pop_ci,tot_hh,tot_fam,tot_pop_ov,...,below2_1_2,disab_1_2,hus_re_1_2,zvhh_1_2,epc_2035,epc_2040,epc_2050,c2040_2050,epc_class,geometry
0,06001401000,06,001,401000,6024,6016,6016,2439,1159,5790,...,1,1,1,1,1,1,1,0,High,"POLYGON ((-122.27871 37.82715, -122.27792 37.8..."
1,06001401300,06,001,401300,3855,3563,3563,1804,457,3768,...,1,1,1,1,1,1,1,0,Higher,"POLYGON ((-122.27339 37.81272, -122.27138 37.8..."


### **3. Organizing Spatial Data for Schools**

#### *a) Explore California Open Data Portal School Data*

In [7]:
# Organize the school datasets
# Sources:
# https://data.ca.gov/dataset/california-private-schools-2023-24
# https://data.ca.gov/dataset/california-public-schools-2023-24

'''
Note: There are will be code strings for each person's unique filepaths. Hide other people's filepath code as a comment by using Ctrl + /.

'''

# Michael's filepath - Read GeoJSON files
# ca_public_schools = gpd.read_file("/content/drive/MyDrive/Urban Informatics Final Project/Data/Schools/Unprocessed Schools Data/California Public Schools 2023-2024.geojson")
# ca_private_schools = gpd.read_file("/content/drive/MyDrive/Urban Informatics Final Project/Data/Schools/Unprocessed Schools Data/California Private Schools 2023-2024.geojson")

# Phoebe's filepath - Read GeoJSON files
ca_public_schools = gpd.read_file("/content/drive/MyDrive/Coursework/Spring 2026/CP 255/Urban Informatics Final Project/Data/Schools/Unprocessed Schools Data/California Public Schools 2023-2024.geojson")
ca_private_schools = gpd.read_file("/content/drive/MyDrive/Coursework/Spring 2026/CP 255/Urban Informatics Final Project/Data/Schools/Unprocessed Schools Data/California Private Schools 2023-2024.geojson")

In [8]:
ca_public_schools.columns

Index(['OBJECTID', 'Year', 'FedID', 'CDSCode', 'CDCode', 'SCode', 'Region',
       'CountyName', 'DistrictName', 'SchoolName', 'SchoolType', 'Status',
       'OpenDate', 'ClosedDate', 'SchoolLevel', 'GradeLow', 'GradeHigh',
       'Charter', 'CharterNum', 'FundingType', 'Virtual', 'Magnet',
       'TitleIStatus', 'DASS', 'AssistStatusESSA', 'Street', 'City', 'Zip',
       'State', 'CongUS', 'SenateCA', 'AssemblyCA', 'Locale', 'Latitude',
       'Longitude', 'EnrollTotal', 'AAcount', 'AApct', 'AIcount', 'AIpct',
       'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount',
       'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct',
       'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct',
       'MIGcount', 'MIGpct', 'SEDCount', 'SEDpct', 'SWDcount', 'SWDpct',
       'FRPMcount', 'FRPMpct', 'CountyCodeGeo', 'CountyNameGeo',
       'DistrictElemCodeGeo', 'DistrictElemNameGeo', 'DistrictHighCodeGeo',
       'DistrictHighNameGeo', 'DistrictUnifi

In [9]:
ca_private_schools.columns

Index(['OBJECTID', 'Join_Count', 'TARGET_FID', 'County', 'District',
       'SchoolName', 'Street', 'City', 'State', 'Zip', 'MailStreet',
       'MailCity', 'MailState', 'MailZip', 'Telephone', 'PrimaryEma',
       'SchoolType', 'Accomm', 'Class', 'LowGrade', 'HighGrade', 'EnrollK',
       'Enroll01', 'Enroll02', 'Enroll03', 'Enroll04', 'Enroll05', 'Enroll06',
       'Enroll07', 'Enroll08', 'Enroll09', 'Enroll10', 'Enroll11', 'Enroll12',
       'TotalEnrol', 'PrevYrGrad', 'ftTeachers', 'ptTeachers', 'Admin',
       'OtherStaff', 'Lat', 'Lon', 'COUNTY1', 'coTest1', 'COUNTY2', 'coTest2',
       'CDS', 'GrK5', 'Gr68', 'Gr912', 'geometry'],
      dtype='object')

In [10]:
# Create lists of variables to keep for each of the two dataframes
# Save this list for the future for race/ethnicity or enrollment levels analysis
public_school_vars = ['Year', 'FedID', 'Region',
       'CountyName', 'DistrictName', 'SchoolName', 'SchoolType', 'Status',
       'OpenDate', 'ClosedDate', 'SchoolLevel', 'GradeLow', 'GradeHigh',
       'Charter', 'Virtual', 'Magnet','TitleIStatus', 'Street', 'City', 'Zip',
       'State', 'Latitude', 'Longitude', 'EnrollTotal', 'AAcount', 'AApct', 'AIcount', 'AIpct',
       'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount',
       'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct',
       'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct',
       'MIGcount', 'MIGpct', 'SEDCount', 'SEDpct', 'SWDcount', 'SWDpct',
       'FRPMcount', 'FRPMpct', 'geometry']

private_school_vars = ['County', 'District','SchoolName', 'Street', 'City', 'State', 'Zip',
                       'Telephone', 'PrimaryEma', 'SchoolType', 'Accomm', 'Class', 'LowGrade', 'HighGrade',
                       'EnrollK', 'Enroll01', 'Enroll02', 'Enroll03', 'Enroll04', 'Enroll05', 'Enroll06', 'Enroll07',
                       'Enroll08', 'Enroll09', 'Enroll10', 'Enroll11', 'Enroll12', 'TotalEnrol',
                       'ftTeachers', 'ptTeachers', 'Admin', 'OtherStaff', 'Lat', 'Lon', 'geometry']

#### *b) Create truncated dataframes to contacenate*

In [11]:
# Create a list of variables for final project
final_public_school_vars = ['CountyName', 'SchoolName', 'City', 'GradeLow', 'GradeHigh', 'Latitude', 'Longitude',
                            'EnrollTotal', 'geometry']

final_private_school_vars = ['County', 'SchoolName', 'City', 'LowGrade', 'HighGrade', 'Lat', 'Lon', 'geometry',
                             'TotalEnrol']

In [12]:
# Create the truncated dataframes
ca_pub_schools = ca_public_schools[final_public_school_vars]
ca_priv_schools = ca_private_schools[final_private_school_vars]

In [13]:
# Rename the column with county information in `ca_pub_schools`
ca_pub_schools = ca_pub_schools.rename(columns={
    'CountyName': 'County'
})

In [14]:
# Rename the `ca_priv_schools` columns to match the public school column names
ca_priv_schools = ca_priv_schools.rename(columns={
    'LowGrade': 'GradeLow',
    'HighGrade': 'GradeHigh',
    'Lat': 'Latitude',
    'Lon': 'Longitude',
    'TotalEnrol': 'EnrollTotal'
})

In [15]:
# Create a new column for each dataframe indicating school type
ca_pub_schools['Type'] = 'Public'
ca_priv_schools['Type'] = 'Private'

In [16]:
# Concacenate the two dataframes
ca_schools = pd.concat([ca_pub_schools, ca_priv_schools])

In [17]:
# Print the shape of all three dataframes
print(ca_pub_schools.shape)
print(ca_priv_schools.shape)
print(ca_schools.shape)

(9996, 10)
(2856, 10)
(12852, 10)


In [18]:
ca_schools.head(2)

,County,SchoolName,City,GradeLow,GradeHigh,Latitude,Longitude,EnrollTotal,geometry,Type
0,Alameda,Envision Academy for Arts & Technology,Oakland,06,12,37.804676,-122.268366,223,POINT (-13610852.265 4551869.704),Public
1,Alameda,Community School for Creative Education,Oakland,KG,08,37.784900,-122.238800,186,POINT (-13607560.992 4549083.796),Public


### **4. Filter for Schools Within EPC Study Areas**

#### *a) Filter School Data for Alameda County Only*

Only keeping Alameda County schools because Tile2Net is only supported in Alameda County.

In [19]:
# Subset the dataframe for Alameda County schools
ac_schools = ca_schools[ca_schools['County'] == 'Alameda']

In [20]:
# Sort the dataframe alphabetically
ac_schools = ac_schools.sort_values('SchoolName', ascending=True)

In [21]:
# Create a dataframe of schools in the cities/unincorporated areas representing our project
# This entails Alameda, Castro Valley, and San Leandro
ac_schools['City'].unique()

array(['Oakland', 'Livermore', 'Alameda', 'Hayward', 'San Leandro',
       'Albany', 'Pleasanton', 'Fremont', 'Union City', 'Berkeley',
       'Emeryville', 'San Lorenzo', 'Newark', 'Piedmont', 'Castro Valley',
       'Dublin', 'Byron', 'Sunol'], dtype=object)

In [22]:
# Print the shape of all dataframes
print(ca_schools.shape)
print(ac_schools.shape)
print(mtc_epcs.shape)

(12852, 10)
(497, 10)
(339, 40)


#### *b) Conduct a Within Spatial Join*

In [23]:
# Get the CRS of `ac_schools`
ac_schools.crs

<Projected CRS: EPSG:3857>
Name: WGS 84 / Pseudo-Mercator
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: World between 85.06°S and 85.06°N.
- bounds: (-180.0, -85.06, 180.0, 85.06)
Coordinate Operation:
- name: Popular Visualisation Pseudo-Mercator
- method: Popular Visualisation Pseudo Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [24]:
# Get CRS of `mtc_epcs`
mtc_epcs.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [25]:
# Adjust the CRS of schools to the `mtc_epcs` CRS
mtc_ac_schools = ac_schools.to_crs(mtc_epcs.crs)

In [26]:
# Check your work
mtc_ac_schools.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [27]:
# Conduct a spatial join
proj_schools_gdf = gpd.sjoin(mtc_ac_schools, mtc_epcs, how='inner', predicate='within')

In [28]:
# Get the shape of the project dataframe
proj_schools_gdf.shape

(150, 50)

### **5. Finalize Project Study Areas**

In [29]:
# Explore the project dataframe of schools
m = proj_schools_gdf.explore(color="blue", name="Schools")

mtc_epcs.explore(
    m=m,
    color="red",
    name="Study Areas"
)
m

#### *a) Create DataFrame for EPC Evaluation*

In [30]:
proj_schools_gdf.columns

Index(['County', 'SchoolName', 'City', 'GradeLow', 'GradeHigh', 'Latitude',
       'Longitude', 'EnrollTotal', 'geometry', 'Type', 'index_right', 'geoid',
       'state_fip', 'county_fip', 'tract', 'tot_pop', 'tot_pop_po',
       'tot_pop_ci', 'tot_hh', 'tot_fam', 'tot_pop_ov', 'pop_over75',
       'pop_poc', 'pop_spfam', 'pop_lep', 'pop_below2', 'pop_disabi',
       'pop_hus_re', 'pop_zvhhs', 'pct_over75', 'pct_poc', 'pct_spfam',
       'pct_lep', 'pct_below2', 'pct_disab', 'pct_hus_re', 'pct_zvhhs',
       'over75_1_2', 'poc_1_2', 'spfam_1_2', 'lep_1_2', 'below2_1_2',
       'disab_1_2', 'hus_re_1_2', 'zvhh_1_2', 'epc_2035', 'epc_2040',
       'epc_2050', 'c2040_2050', 'epc_class'],
      dtype='object')

In [31]:
# Create a dataframe with important sites attributes
potential_sites = proj_schools_gdf.groupby('geoid').agg(
    SchoolCount=('EnrollTotal', 'size'),
    EnrollTotal=('EnrollTotal', 'sum'),
    pct_poc=('pct_poc', 'first'),
    pct_spfam=('pct_spfam', 'first'),
    pct_lep=('pct_lep', 'first'),
    pct_below2=('pct_below2', 'first'),
    pct_zvhhs=('pct_zvhhs', 'first')
).reset_index()

In [32]:
# Count number of EPCs in the project dataframe
proj_schools_gdf['geoid'].nunique()

75

In [33]:
# Determine shape of potential sites dataframe
potential_sites.shape

(75, 8)

In [34]:
# Print first few rows of potential_sites
potential_sites.head(3)

,geoid,SchoolCount,EnrollTotal,pct_poc,pct_spfam,pct_lep,pct_below2,pct_zvhhs
0,06001401000,2,629,0.683599,0.315789,0.036960,0.295878,0.264453
1,06001401300,1,57,0.617121,0.170678,0.091030,0.467583,0.333149
2,06001401400,2,274,0.761025,0.126806,0.106645,0.586701,0.272208


In [35]:
# Print summary statistics for potential sites
potential_sites.describe()

,SchoolCount,EnrollTotal,pct_poc,pct_spfam,pct_lep,pct_below2,pct_zvhhs
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000
mean,2.000000,772.213333,0.824794,0.252749,0.152609,0.433731,0.172308
std,1.375461,718.438496,0.121729,0.096745,0.081560,0.105795,0.124173
min,1.000000,1.000000,0.323998,0.008996,0.009942,0.285221,0.013457
25%,1.000000,341.500000,0.752482,0.201711,0.100227,0.335485,0.090474
50%,1.000000,589.000000,0.857856,0.234704,0.144078,0.430425,0.158409
75%,2.000000,914.500000,0.908698,0.299098,0.206370,0.519503,0.206596
max,6.000000,3283.000000,0.981045,0.601562,0.356500,0.679393,0.631605


#### *b) Finalize Project Areas*

##### **i) Finalize Project Areas**

In [36]:
# Get a copy of census_tracts to prepare epc_study areas
epc_study_areas = mtc_epcs.copy()

In [37]:
# Correct the variable type of the `tract` column
epc_study_areas['tract'] = epc_study_areas['tract'].astype(float).astype(int)

In [38]:
# Filter the dataframe to only include our EPCs of interest
epc_study_areas = epc_study_areas[
    (epc_study_areas['tract'] == 432400) | # San Leandro
    (epc_study_areas['tract'] == 428700) | # Alameda
    (epc_study_areas['tract'] == 430500) # Castro Valley
]

In [39]:
# Create a dataframe for each individual EPC
# Alameda
alameda_epc_gdf = epc_study_areas[epc_study_areas['tract'] == 428700]

# Castro Valley
cv_epc_gdf = epc_study_areas[epc_study_areas['tract'] == 430500]

# San Leandro
sl_epc_gdf = epc_study_areas[epc_study_areas['tract'] == 432400]

##### **ii) Finalize Schools Within Project Areas**

In [40]:
# Make a copy of the dataframe
final_proj_schools_gdf = proj_schools_gdf.copy()

In [41]:
# Correct the variable type of the `tract` column
final_proj_schools_gdf['geoid'] = final_proj_schools_gdf['geoid'].astype(float).astype(int)

In [42]:
# Filter the dataframe to only include our EPCs of interest
final_proj_schools_gdf = final_proj_schools_gdf[
    (final_proj_schools_gdf['geoid'] == 6001432400) | # San Leandro
    (final_proj_schools_gdf['geoid'] == 6001428700) | # Alameda
    (final_proj_schools_gdf['geoid'] == 6001430500) # Castro Valley
]

In [43]:
# Print the final schools dataframe
final_proj_schools_gdf

,County,SchoolName,City,GradeLow,GradeHigh,Latitude,Longitude,EnrollTotal,geometry,Type,...,lep_1_2,below2_1_2,disab_1_2,hus_re_1_2,zvhh_1_2,epc_2035,epc_2040,epc_2050,c2040_2050,epc_class
4,Alameda,Alameda County Juvenile Hall/Court,San Leandro,KG,12,37.715956,-122.118237,44,POINT (-122.11824 37.71596),Public,...,0,1,1,1,0,0,0,1,1,High
16,Alameda,Alameda Science and Technology Institute,Alameda,09,12,37.781464,-122.280694,167,POINT (-122.28069 37.78146),Public,...,1,1,0,0,1,1,1,1,0,High
309,Alameda,Garfield Elementary,San Leandro,KG,05,37.703705,-122.185413,338,POINT (-122.18541 37.7037),Public,...,0,1,0,0,0,1,1,1,0,High
17,Alameda,Ruby Bridges Elementary,Alameda,KG,05,37.782749,-122.284969,434,POINT (-122.28497 37.78275),Public,...,1,1,0,0,1,1,1,1,0,High
148,Alameda,Seneca Family of Agencies - James Baldwin Academy,San Leandro,Fifth Grade,Twelfth Grade,37.708557,-122.111762,41,POINT (-122.11176 37.70856),Private,...,0,1,1,1,0,0,0,1,1,High


In [44]:
# Print summary statistics of the final schools dataframe
final_proj_schools_gdf.describe()

,Latitude,Longitude,EnrollTotal,index_right,geoid,tot_pop,tot_pop_po,tot_pop_ci,tot_hh,tot_fam,...,spfam_1_2,lep_1_2,below2_1_2,disab_1_2,hus_re_1_2,zvhh_1_2,epc_2035,epc_2040,epc_2050,c2040_2050
count,5.000000,5.000000,5.0000,5.00000,5.000000e+00,5.000000,5.00000,5.000000,5.000000,5.000000,...,5.0,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.0,5.000000
mean,37.738486,-122.196215,204.8000,71.80000,6.001430e+09,5529.800000,5204.00000,5188.600000,1756.800000,1216.200000,...,1.0,0.400000,1.0,0.400000,0.400000,0.400000,0.600000,0.600000,1.0,0.400000
std,0.040061,0.084178,176.3454,0.83666,1.542077e+03,979.255176,1017.04056,1132.100172,334.491704,179.004749,...,0.0,0.547723,0.0,0.547723,0.547723,0.547723,0.547723,0.547723,0.0,0.547723
min,37.703705,-122.284969,41.0000,71.00000,6.001429e+09,4484.000000,4142.00000,4019.000000,1408.000000,1023.000000,...,1.0,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000
25%,37.708557,-122.280694,44.0000,71.00000,6.001429e+09,4484.000000,4142.00000,4019.000000,1408.000000,1023.000000,...,1.0,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000
50%,37.715956,-122.185413,167.0000,72.00000,6.001430e+09,6049.000000,5661.00000,5661.000000,1822.000000,1295.000000,...,1.0,0.000000,1.0,0.000000,0.000000,0.000000,1.000000,1.000000,1.0,0.000000
75%,37.781464,-122.118237,338.0000,72.00000,6.001430e+09,6049.000000,5661.00000,5661.000000,2073.000000,1370.000000,...,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000
max,37.782749,-122.111762,434.0000,73.00000,6.001432e+09,6583.000000,6414.00000,6583.000000,2073.000000,1370.000000,...,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000


In [45]:
# Create a dataframe for each individual EPC
# Alameda
alameda_epc_schools = final_proj_schools_gdf[final_proj_schools_gdf['geoid'] == 6001428700]

# Castro Valley
cv_epc_schools = final_proj_schools_gdf[final_proj_schools_gdf['geoid'] == 6001430500]

# San Leandro
sl_epc_schools = final_proj_schools_gdf[final_proj_schools_gdf['geoid'] == 6001432400]

In [46]:
# Explore the project dataframe of schools in final study areas
m = final_proj_schools_gdf.explore(color="blue", name="Schools")

epc_study_areas.explore(
    m=m,
    color="red",
    name="Study Areas"
)
m

### **6. Export Project Files**

In [47]:
# Export the cleaned GeoDataFrame

# Study Areas
epc_study_areas.to_file("CP 255 Final Project EPCs.geojson", driver="GeoJSON")
alameda_epc_gdf.to_file("CP 255 Final Project Alameda EPC.geojson", driver="GeoJSON")
cv_epc_gdf.to_file("CP 255 Final Project Castro Valley EPC.geojson", driver="GeoJSON")
sl_epc_gdf.to_file("CP 255 Final Project San Leandro EPC.geojson", driver="GeoJSON")

# Schools
final_proj_schools_gdf.to_file("CP 255 Final Project Schools.geojson", driver="GeoJSON")
alameda_epc_schools.to_file("CP 255 Final Project Alameda Schools.geojson", driver="GeoJSON")
cv_epc_schools.to_file("CP 255 Final Project Castro Valley Schools.geojson", driver="GeoJSON")
sl_epc_schools.to_file("CP 255 Final Project San Leandro Schools.geojson", driver="GeoJSON")

from google.colab import files
# files.download('CP 255 Final Project EPCs.geojson')
# files.download("CP 255 Final Project Alameda EPC.geojson")
# files.download("CP 255 Final Project Castro Valley EPC.geojson")
# files.download("CP 255 Final Project San Leandro EPC.geojson")

# files.download('CP 255 Final Project Schools.geojson')
# files.download("CP 255 Final Project Alameda Schools.geojson")
# files.download("CP 255 Final Project Castro Valley Schools.geojson")
# files.download("CP 255 Final Project San Leandro Schools.geojson")

### **OPTIONAL: Public Schools Demographics**

#### *a) Filter Public School Data for Alameda County*

Only keeping Alameda County schools because Tile2Net is only supported in Alameda County.

In [48]:
# Subset the dataframe for Alameda County schools
ac_schools_public = ca_public_schools[ca_pub_schools['County'] == 'Alameda']

In [49]:
ac_schools_public = ac_schools_public[public_school_vars]

In [50]:
ac_schools_public.shape

(370, 57)

#### *b) Conduct a Within Spatial Join*

In [51]:
# Adjust the CRS of schools to the `mtc_epcs` CRS
mtc_ac_schools_public = ac_schools_public.to_crs(mtc_epcs.crs)

In [52]:
# Conduct a spatial join
proj_schools_public_gdf = gpd.sjoin(mtc_ac_schools_public, mtc_epcs, how='inner', predicate='within')

In [53]:
# Get the number of schools within potential EPC sites
proj_schools_public_gdf.shape

(122, 97)

In [54]:
proj_schools_public_gdf.head(1).T

,0
Year,2023-24
FedID,060161410947
Region,04
CountyName,Alameda
DistrictName,Alameda County Office of Education
...,...
epc_2035,1
epc_2040,1
epc_2050,1
c2040_2050,0


#### *c) Create DataFrame for EPC Evaluation*

In [55]:
# Create a dataframe with important sites attributes
potential_public_sites = proj_schools_public_gdf.groupby('geoid').agg(
    SchoolCount=('EnrollTotal', 'size'),
    EnrollTotal=('EnrollTotal', 'sum'),
    SEDTotal=('SEDCount', 'sum'),
    tot_pop=('tot_pop', 'first'),
    pct_poc=('pct_poc', 'first'),
    pct_spfam=('pct_spfam', 'first'),
    pct_lep=('pct_lep', 'first'),
    pct_below2=('pct_below2', 'first'),
    pct_zvhhs=('pct_zvhhs', 'first')
).reset_index()

In [56]:
# Count number of EPCs in the project dataframe
proj_schools_public_gdf['geoid'].nunique()

66

In [57]:
# Determine shape of potential sites dataframe
potential_public_sites.shape

(66, 10)

In [58]:
# Create a variable with the percentage of students
potential_public_sites['pct_students'] = potential_public_sites['EnrollTotal'] / potential_public_sites['tot_pop']

In [59]:
# Create a variable with the percentage of SED students
potential_public_sites['sed_share'] = potential_public_sites['SEDTotal'] / potential_public_sites['EnrollTotal']

In [60]:
# Create a variable with average school size
potential_public_sites['avg_school_size'] = potential_public_sites['EnrollTotal'] / potential_public_sites['SchoolCount']

In [61]:
# Run summary statistics
potential_public_sites[
    (potential_public_sites['SchoolCount'] > 1) &
    (potential_public_sites['pct_students'] > 0.2) &
    (potential_public_sites['pct_zvhhs'] > 0.18)
    # (potential_public_sites['sed_share'] > 0.8) &
    # (potential_public_sites['SEDTotal'] > 1000)
].sort_values('sed_share', ascending = False)

,geoid,SchoolCount,EnrollTotal,SEDTotal,tot_pop,pct_poc,pct_spfam,pct_lep,pct_below2,pct_zvhhs,pct_students,sed_share,avg_school_size
5,06001402500,3,579,569,1786,0.912094,0.601562,0.021654,0.638298,0.539548,0.324188,0.982729,193.00
30,06001408800,5,2125,2076,7054,0.949107,0.418909,0.215851,0.665904,0.316194,0.301248,0.976941,425.00
37,06001409800,4,1435,1352,3519,0.809321,0.318976,0.038087,0.299914,0.187309,0.407786,0.942160,358.75
14,06001406000,5,1036,888,3344,0.818182,0.008996,0.310638,0.532297,0.234812,0.309809,0.857143,207.20
9,06001403300,4,1625,1283,4240,0.766509,0.129300,0.282686,0.430425,0.367118,0.383255,0.789538,406.25
3,06001401600,2,1130,526,2269,0.678713,0.401760,0.105263,0.435875,0.180876,0.498017,0.465487,565.00


In [62]:
# potential_sites['pct_poc_wt'] = (
#     proj_schools_gdf.groupby('geoid').apply(
#         lambda x: (x['pct_poc'] * x['EnrollTotal']).sum() / x['EnrollTotal'].sum()
#     )
# )

In [63]:
# Explore the project dataframe of schools
m = proj_schools_gdf.explore(color="blue", name="Schools")

mtc_epcs.explore(
    m=m,
    color="red",
    name="Study Areas"
)
m